In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal

import os
import pickle

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

class SACActor(nn.Module):
    def __init__(self):
        super(SACActor, self).__init__()
        
        # 影像特徵提取
        self.map_embed = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), # 40x48
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), # 20x24
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), # 10x12
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(128 * 10 * 12, 512), 
            nn.ReLU(),
            nn.Linear(512, 64), 
            nn.ReLU()
        )
        
        # 儀表板特徵提取
        self.dashboard_embed = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU()
        )
        
        # 獨立的輸出頭 (不可共用 nn.Linear)
        self.mu_head = nn.Linear(128, 3)
        self.log_std_head = nn.Linear(128, 3)

    def forward(self, map_img, dashboard_vec):
        # 影像分支
        map_feat = self.map_embed(map_img)
        # 數值分支
        dash_feat = self.dashboard_embed(dashboard_vec)
        
        # 特徵對齊拼接 (維度 1)
        embedding = torch.cat([map_feat, dash_feat], dim=1) # (Batch, 128)
        
        mu = self.mu_head(embedding)
        log_std = self.log_std_head(embedding)
        log_std = torch.clamp(log_std, -20, 2)
        return mu, log_std

    def sample(self, map_img, dashboard_vec):
        # 呼叫修正後的 forward
        mu, log_std = self.forward(map_img, dashboard_vec)
        std = log_std.exp()
        dist = Normal(mu, std)
        x_t = dist.rsample() 
        
        action_tanh = torch.tanh(x_t)
        
        # 計算 Log Prob 並進行 tanh 修正
        log_prob = dist.log_prob(x_t) - torch.log(1 - action_tanh.pow(2) + 1e-6)
        
        # 分離與縮放動作 [Steer, Gas, Brake]
        steer = action_tanh[:, 0:1]
        gas_brake = (action_tanh[:, 1:] + 1.0) / 2.0
        action = torch.cat([steer, gas_brake], dim=1)
        
        # 針對 Gas/Brake 維度進行線性縮放修正 (log 2)
        # 這裡用一個簡單的加法處理
        correction = torch.zeros_like(log_prob)
        correction[:, 1:] = torch.log(torch.tensor(2.0))
        log_prob += correction

        return action, log_prob.sum(1, keepdim=True)
    
class QNetwork(nn.Module):
    def __init__(self):
        super(QNetwork, self).__init__()
        
        # 影像特徵提取
        self.map_embed = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1), # 40x48
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1), # 20x24
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), # 10x12
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(128 * 10 * 12, 512), 
            nn.ReLU(),
            nn.Linear(512, 64), 
            nn.ReLU()
        )
        
        # 儀表板特徵提取
        self.dashboard_embed = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU()
        )
        
        # 動作特徵提取
        self.action_embed = nn.Sequential(
            nn.Linear(3, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU()
        )
        
        # Q 值融合頭 (輸入：影像 64 + 數值 64 + 動作 64 = 131)
        self.q_head = nn.Sequential(
            nn.Linear(64 + 64 + 64, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1) # 輸出單一標量 Q 值
        )

    def forward(self, map_img, dash_vec, action):
        m_feat = self.map_embed(map_img)
        d_feat = self.dashboard_embed(dash_vec)
        a_feat = self.action_embed(action)
        
        return self.q_head(torch.cat([m_feat, d_feat, a_feat], dim=1))

class SACDoubleCritic(nn.Module):
    def __init__(self):
        super(SACDoubleCritic, self).__init__()
        # SAC 核心：建立兩個完全獨立的 Q 網路
        self.Q1 = QNetwork()
        self.Q2 = QNetwork()

    def forward(self, map_img, dash_vec, action):
        q1 = self.Q1(map_img, dash_vec, action)
        q2 = self.Q2(map_img, dash_vec, action)
        return q1, q2
    
import random
from collections import deque

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state_img, dash_vec, action, reward, next_state_img, next_dash_vec, done):
        self.buffer.append((state_img, dash_vec, action, reward, next_state_img, next_dash_vec, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        state_img, dash_vec, action, reward, next_state_img, next_dash_vec, done = zip(*batch)
        return (np.array(state_img), np.array(dash_vec), np.array(action, dtype=np.float32), 
                np.array(reward, dtype=np.float32), np.array(next_state_img), 
                np.array(next_dash_vec), np.array(done, dtype=np.float32))

    def __len__(self):
        return len(self.buffer)
    
def preprocess(state_img, dashboard_vec, device):
    # 影像: 標準 0~1 歸一化
    img = torch.FloatTensor(state_img.copy()).permute(2, 0, 1).unsqueeze(0) / 255.0
    
    # 建立一個與 dashboard_vec 同形狀的 scale 向量
    # 速度上限 100, 輪速上限 230
    scales = torch.FloatTensor([100.0, 230.0, 230.0, 230.0, 230.0, 1.0, 1.0, 1.0]).to(device)
    
    # 轉換為 Tensor 並直接除以各自的上限
    dash = torch.FloatTensor(dashboard_vec).to(device) / scales
    dash = dash.unsqueeze(0) # 補上 Batch 維度
    
    return img.to(device), dash

class SACAgent:
    def __init__(self, device):
        self.device = device
        self.actor = SACActor().to(device)
        self.critic = SACDoubleCritic().to(device)
        self.target_critic = SACDoubleCritic().to(device)
        self.target_critic.load_state_dict(self.critic.state_dict())
        
        # 優化器
        self.actor_opt = torch.optim.Adam(self.actor.parameters(), lr=3e-4)
        self.critic_opt = torch.optim.Adam(self.critic.parameters(), lr=3e-4)
        
        # 自動溫度調整 (Entropy Alpha)
        self.target_entropy = -3.0 # 目錄維度 -action_dim
        self.log_alpha = torch.zeros(1, requires_grad=True, device=device)
        self.alpha_opt = torch.optim.Adam([self.log_alpha], lr=3e-4)
        
        self.gamma = 0.99
        self.tau = 0.005

    def update_parameters(self, buffer, batch_size):
        # 取得資料與權重
        s_img, s_dash, a, r, ns_img, ns_dash, d, idxs, is_weights = buffer.sample(batch_size)
        
        # 轉為 Tensor
        device = self.device
        s_img = torch.FloatTensor(s_img).permute(0, 3, 1, 2).to(device) / 255.0
        ns_img = torch.FloatTensor(ns_img).permute(0, 3, 1, 2).to(device) / 255.0
        scales = torch.FloatTensor([100.0, 230.0, 230.0, 230.0, 230.0, 1.0, 1.0, 1.0]).to(device)
        s_dash, ns_dash = torch.FloatTensor(s_dash).to(device)/scales, torch.FloatTensor(ns_dash).to(device)/scales
        a, r, d = torch.FloatTensor(a).to(device), torch.FloatTensor(r).unsqueeze(1).to(device), torch.FloatTensor(d).unsqueeze(1).to(device)
        
        # 將 IS Weights 轉為 Tensor
        is_weights = torch.FloatTensor(is_weights).unsqueeze(1).to(device)

        with torch.no_grad():
            next_action, next_log_prob = self.actor.sample(ns_img, ns_dash)
            q1_t, q2_t = self.target_critic(ns_img, ns_dash, next_action)
            target_v = torch.min(q1_t, q2_t) - self.log_alpha.exp() * next_log_prob
            target_q = r + (1 - d) * self.gamma * target_v

        # 更新 Critic
        curr_q1, curr_q2 = self.critic(s_img, s_dash, a)
        
        # 計算 TD-Error 以便更新優先級
        td_error1 = curr_q1 - target_q
        td_error2 = curr_q2 - target_q
        
        # 更新 Buffer 優先級 (取兩者平均的絕對值)
        new_priorities = (torch.abs(td_error1) + torch.abs(td_error2)).detach().cpu().numpy() / 2.0
        buffer.update_priorities(idxs, new_priorities)

        # 使用 IS Weights 修正 Loss (注意 reduction='none')
        critic_loss = (is_weights * F.mse_loss(curr_q1, target_q, reduction='none')).mean() + \
                      (is_weights * F.mse_loss(curr_q2, target_q, reduction='none')).mean()
        
        self.critic_opt.zero_grad(); critic_loss.backward(); self.critic_opt.step()

        # 更新 Actor
        new_a, log_prob = self.actor.sample(s_img, s_dash)
        q1_new, q2_new = self.critic(s_img, s_dash, new_a)
        actor_loss = (is_weights * (self.log_alpha.exp() * log_prob - torch.min(q1_new, q2_new))).mean()
        self.actor_opt.zero_grad(); actor_loss.backward(); self.actor_opt.step()

        # 更新 Alpha
        alpha_loss = -(self.log_alpha * (log_prob + self.target_entropy).detach()).mean()
        self.alpha_opt.zero_grad(); alpha_loss.backward(); self.alpha_opt.step()

        # Soft Update
        for t, s in zip(self.target_critic.parameters(), self.critic.parameters()):
            t.data.copy_(t.data * (1.0 - self.tau) + s.data * self.tau)
            
        return new_priorities.mean()

    def save_checkpoint(self, checkpoint_path, episode, now_score, best_score):
        checkpoint = {
            'episode': episode,
            'now_score': now_score,
            'best_score': best_score,
            'actor_state_dict': self.actor.state_dict(),
            'critic_state_dict': self.critic.state_dict(),
            'target_critic_state_dict': self.target_critic.state_dict(),
            'actor_opt_state_dict': self.actor_opt.state_dict(),
            'critic_opt_state_dict': self.critic_opt.state_dict(),
            'log_alpha': self.log_alpha,
            'alpha_opt_state_dict': self.alpha_opt.state_dict()
        }
        torch.save(checkpoint, checkpoint_path)
    
    def load_checkpoint(self, checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        self.actor.load_state_dict(checkpoint['actor_state_dict'])
        self.critic.load_state_dict(checkpoint['critic_state_dict'])
        self.target_critic.load_state_dict(checkpoint['target_critic_state_dict'])
        self.actor_opt.load_state_dict(checkpoint['actor_opt_state_dict'])
        self.critic_opt.load_state_dict(checkpoint['critic_opt_state_dict'])
        self.log_alpha.data.copy_(checkpoint['log_alpha'])
        self.alpha_opt.load_state_dict(checkpoint['alpha_opt_state_dict'])
        return checkpoint['episode'], checkpoint['now_score'], checkpoint['best_score']

# 讀取函數
def load_buffer(buffer_obj, path):
    if os.path.exists(path):
        with open(path, 'rb') as f:
            buffer_obj.buffer = pickle.load(f)
        print(f"成功讀取 Buffer，現有經驗：{len(buffer_obj.buffer)}")

env = gym.make("CarRacing-v3", render_mode="human", domain_randomize=False, continuous=True, max_episode_steps=1000, lap_complete_percent=1.0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

agent = SACAgent(device)
batch_size = 64

checkpoint_path = input("checkpoint path:")
start_ep = 0
now_score = 0
best_score = float('-inf')
if (os.path.exists(checkpoint_path)):
  start_ep, now_score, best_score = agent.load_checkpoint(checkpoint_path)
  start_ep += 1
  print(start_ep, now_score)

for episode in range(start_ep, 1000000000000000000):
    state, info = env.reset()
    # 在迴圈中 reset 之後加入這行
    print(f"Episode {episode} 初始地圖校驗碼: {state.sum()}")
    for t in range(50):
        env.step(np.array([0, 0, 0]))
    state = state[2:82]
    dashboard = np.zeros(8)
    episode_reward = 0
    stay_neg = 0
    
    for t in range(5000):
        # 選擇動作
        img_t, dash_t = preprocess(state, dashboard, device)
        with torch.no_grad():
            action, _ = agent.actor.sample(img_t, dash_t)
        action = action.cpu().numpy()[0]

        # 執行動作
        next_state, reward, done, truncated, info = env.step(action)
        next_state = next_state[2:82]
        
        # 更新物理量 (Dashboard)
        car = env.unwrapped.car
        next_dashboard = np.concat((np.array([np.linalg.norm(car.hull.linearVelocity)] + [w.omega for w in car.wheels]), action))
        
        state, dashboard = next_state, next_dashboard
        episode_reward += reward

        if done or truncated:
            break


c:\miniconda3\envs\AIracecar\lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
C:\Temp\ipykernel_3384\4009126014.py:279: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are 

947001 283.0
Episode 947001 初始地圖校驗碼: 1199281
